# 01 - Data Cleaning

This notebook loads the raw credit card transactions file and works through every cleaning check, one step at a time.

The raw file in `data/raw` is only ever **read** here, never changed. Any cleaned output is written to `data/staging` or `data/processed`.

## Step 1: Load the data and look at it

`pandas` is a library for working with tables of data. We call it `pd`, the name almost everyone uses for it.

In [1]:
import pandas as pd

A quick check of which Python is running this notebook. `sys` is a module built into Python that describes the running program, and `sys.executable` is the location of the Python program itself. It should point inside this project's `.venv` folder, which proves the notebook is using the project's own pinned libraries. `pd.__version__` shows the pandas version, which should match `requirements.txt`.

In [2]:
import sys

print(sys.executable)
print(pd.__version__)

C:\Users\RevathyHarichandran\Desktop\portfolio projects\Fraud_detection\.venv\Scripts\python.exe
3.0.5


By default pandas hides some columns behind `...` when a table is wide. This dataset has 31 columns, so we tell pandas to show all of them. `None` means "no limit".

In [3]:
pd.set_option("display.max_columns", None)

Read the CSV file into a DataFrame (a table in memory) called `transactions`.

The path starts with `../` because this notebook lives inside the `notebooks` folder, so we first step up one folder to the project folder, then go into `data/raw`.

`read_csv` defaults being relied on: values are separated by commas, the first line holds the column names, and pandas guesses each column's type from its contents.

In [4]:
transactions = pd.read_csv("../data/raw/creditcard.csv")

`shape` gives two numbers: (number of rows, number of columns).

In [5]:
print(transactions.shape)

(284807, 31)


The name of every column, in order.

In [6]:
print(list(transactions.columns))

['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


The first rows of the table. With nothing in the brackets, `head()` shows 5 rows by default.

In [7]:
transactions.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Step 2: Add a stable row ID

The numbers on the left of the table (0, 1, 2, ...) are pandas' **index**, each row's current position. Positions change once rows are removed, so we copy them into a real column, `row_id`, that stays attached to its row forever. Because it comes from the untouched raw file, every rerun gives each transaction the same ID.

`row_id` starts at 0 so it matches pandas' own row numbers for the raw file (row_id 5 is the row `head()` shows as 5). It is named `row_id`, not `transaction_id`, because it is our label for a position in the original file, not an ID from the bank.

First, count the rows. `len()` gives the number of rows in the table.

In [8]:
number_of_rows = len(transactions)
print(number_of_rows)

284807


`range(number_of_rows)` produces the whole numbers 0, 1, 2, ... up to one less than the number of rows. `insert` then adds them as a new column: the `0` puts it in position 0 (the far left), `"row_id"` is the column's name, and `row_ids` are the values.

In [9]:
row_ids = range(number_of_rows)
transactions.insert(0, "row_id", row_ids)

Check it worked:
- `is_unique` should be `True`, meaning no two rows share an ID.
- The smallest ID (`min`) should be 0 and the largest (`max`) one less than the number of rows. Together with `is_unique`, that means every number from 0 to the last one is used exactly once.
- The shape should now show 32 columns.

In [10]:
print(transactions["row_id"].is_unique)
print(transactions["row_id"].min())
print(transactions["row_id"].max())
print(transactions.shape)

True
0
284806
(284807, 32)


`row_id` should now be the first column (same `head()` as in Step 1).

In [11]:
transactions.head()

,row_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


Save this version to `data/staging`, so later cleaning steps can start from it. `data/raw` is never written to.

`index=False` stops pandas writing its position numbers as an extra, unnamed column. By default (`index=True`) it would write them, which would just duplicate `row_id`.

In [12]:
transactions.to_csv("../data/staging/creditcard_with_row_id.csv", index=False)

## Step 3: Check data types and missing values

Every column has a **data type** (dtype): `int64` for whole numbers, `float64` for decimals, and `str` for text. Expected: `row_id` and `Class` as `int64`, everything else as `float64`. `Class` is the one to watch, since its values had quote marks in the raw file; if pandas had read them as text it would show `str`.

First, every column's type, then a count of how many columns have each type, so an odd one out is easy to spot.

In [13]:
print(transactions.dtypes)
print()
print(transactions.dtypes.value_counts())

row_id      int64
Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

float64    30
int64       2
Name: count, dtype: int64


Is `Class` really binary? `unique()` lists every distinct value in the column. It should be exactly `0` and `1`.

In [14]:
print(transactions["Class"].unique())

[0 1]


Is `Time` whole seconds? It is stored as a decimal, so we check. `%` gives the remainder after dividing, so `Time % 1` is the part after the decimal point, which is 0 for a whole number. `!= 0` marks each row `True` if it has a fractional part, and `.sum()` counts the `True`s (pandas counts `True` as 1 and `False` as 0). This matters for the tie check later.

In [15]:
has_fraction = transactions["Time"] % 1 != 0
print(has_fraction.sum())

0


Missing values. `isna()` marks every empty cell (shown by pandas as `NaN`, "not a number") as `True`, and `.sum()` counts them for each column. Adding those counts together gives one grand total for the whole table.

Limit: this only finds truly empty cells. It can't spot a stand-in value such as `0` or `-1` being used to mean "unknown"; the zero-Amount check in the next step covers the most likely case of that here.

In [16]:
missing_per_column = transactions.isna().sum()
print(missing_per_column)
print()
print(missing_per_column.sum())

row_id    0
Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

0


## Step 4: Check Amount for negative and zero values

A negative amount would signal a real data problem. First, count rows with `Amount < 0`, then show the smallest Amount in the file as a second check: if the smallest isn't below 0, nothing is.

In [17]:
is_negative = transactions["Amount"] < 0
print(is_negative.sum())
print(transactions["Amount"].min())

0
0.0


Now count rows where Amount is exactly zero. `==` means "is equal to" (a single `=` means "store this value", which is different). A zero in the file is written as a plain `0`, which the computer stores as exactly 0, so the exact comparison is safe here.

In [18]:
is_zero = transactions["Amount"] == 0
print(is_zero.sum())

1825


Look at some of the zero-amount rows. Putting the `True`/`False` column inside square brackets keeps only the rows marked `True`; this is called **boolean filtering**. `head()` then shows the first 5 of them (its default), with `row_id` showing exactly which original transactions they are.

In [19]:
zero_amount_rows = transactions[is_zero]
zero_amount_rows.head()

,row_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
383,383,282.0,-0.356466,0.725418,1.971749,0.831343,0.369681,-0.107776,0.751610,-0.120166,-0.420675,-0.059943,-0.508270,0.425506,0.414309,-0.698375,-1.465349,-0.119009,-0.144735,-1.332221,-1.547440,-0.133602,0.020804,0.424312,-0.015989,0.466754,-0.809962,0.657334,-0.043150,-0.046401,0.0,0
514,514,380.0,-1.299837,0.881817,1.452842,-1.293698,-0.025105,-1.170103,0.861610,-0.193934,0.592001,0.241979,0.144973,-0.583891,-1.595345,0.032613,0.752834,-0.094286,-0.163427,-1.111176,-1.124025,0.065979,-0.272563,-0.360853,0.223911,0.598930,-0.397705,0.637141,0.234872,0.021379,0.0,0
534,534,403.0,1.237413,0.512365,0.687746,1.693872,-0.236323,-0.650232,0.118066,-0.230545,-0.808523,0.511284,-0.178159,0.762909,1.700923,-0.133861,0.402418,0.684668,-0.609395,-0.704277,-0.720726,-0.006716,-0.077543,-0.178220,0.038722,0.471218,0.289249,0.871803,-0.066884,0.012986,0.0,0
541,541,406.0,-2.312227,1.951992,-1.609851,3.997906,-0.522188,-1.426545,-2.537387,1.391657,-2.770089,-2.772272,3.202033,-2.899907,-0.595222,-4.289254,0.389724,-1.140747,-2.830056,-0.016822,0.416956,0.126911,0.517232,-0.035049,-0.465211,0.320198,0.044519,0.177840,0.261145,-0.143276,0.0,1
575,575,430.0,-1.860258,-0.629859,0.966570,0.844632,0.759983,-1.481173,-0.509681,0.540722,-0.733623,-0.371622,0.859741,0.372609,-1.240185,0.998391,-0.346387,-0.391679,0.348289,0.282125,1.165893,0.320450,0.268028,0.125515,-0.225029,0.586664,-0.031598,0.570168,-0.043007,-0.223739,0.0,0


Are the zero-amount rows fraud or genuine? `value_counts()` counts how many rows have each distinct value in a column. If zero-amount rows are fraud far more often than the dataset overall, that is a pattern worth knowing, not just an oddity.

In [20]:
print(zero_amount_rows["Class"].value_counts())

Class
0    1798
1      27
Name: count, dtype: int64


Decision: the zero-amount rows are **kept**. They are real transactions with a real `Class` label, they include 27 of the scarce fraud cases, and removing them would also remove whatever signal "zero amount" carries.

## Step 5: Check Time across the two-day window

The first and last Time values should be 0 and close to 172,800 (two days in seconds). Dividing by 3,600 (seconds in an hour) gives the span in hours, which should be close to 48.

In [21]:
print(transactions["Time"].min())
print(transactions["Time"].max())

time_span_hours = transactions["Time"].max() / 3600
print(time_span_hours)

0.0
172792.0
47.99777777777778


Is the file already in time order? `is_monotonic_increasing` is `True` if Time never goes down from one row to the next. If so, `row_id` order and time order are the same thing.

Then, any big gaps? `diff()` gives, for each row, how many seconds passed since the row before it (the first row gets an empty value, since nothing comes before it). The largest gap is the longest stretch with no transactions at all.

In [22]:
print(transactions["Time"].is_monotonic_increasing)

time_gaps = transactions["Time"].diff()
print(time_gaps.max())

True
32.0


### Ties: transactions sharing the same Time value

`nunique()` counts how many **different** Time values appear. If it is much smaller than the number of rows, many rows share a second.

`duplicated(keep=False)` marks every row whose Time value also appears on at least one other row. The default, `keep="first"`, would leave the first row of each tied group unmarked and only mark the repeats, undercounting what we want to know.

In [23]:
print(transactions["Time"].nunique())

shares_time = transactions["Time"].duplicated(keep=False)
print(shares_time.sum())

124592
239644


How big do the tied groups get? `value_counts()` counts the transactions in each second, sorted busiest first by default. The largest group is the worst case for a split: the most transactions that could sit right on a cut between training, validation and test.

In [24]:
rows_per_second = transactions["Time"].value_counts()
print(rows_per_second.max())
print()
print(rows_per_second.head())

36

Time
163152.0    36
64947.0     26
68780.0     25
3767.0      21
3770.0      20
Name: count, dtype: int64


Decision: about 84% of transactions share their second with another, but the largest group is only 36 transactions. When the data is split by time in Phase 3, cuts are made **only between two different Time values**, so every transaction from the same second lands in the same group. Group sizes may be off target by at most 36 rows per cut, and the rule "everything in training happened no later than everything in validation, and validation no later than test" stays completely true.

## Step 6: Are V1 to V28 centred near zero? Does any column never change?

PCA subtracts each original column's **mean** (average) before blending, so every V column's mean should be zero or extremely close to it.

First, build a list of the 28 V column names with a `for` loop: start with an empty list, go through the numbers 1 to 28 (`range(1, 29)` stops *before* 29), glue "V" onto each number (`str()` turns the number into text), and add it to the list with `append`.

In [25]:
v_columns = []
for number in range(1, 29):
    v_columns.append("V" + str(number))

print(v_columns)

['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28']


The mean of each V column. Putting the list inside square brackets selects just those 28 columns.

In [26]:
v_means = transactions[v_columns].mean()
print(v_means)

V1     1.175161e-15
V2     3.384974e-16
V3    -1.379537e-15
V4     2.094852e-15
V5     1.021879e-15
V6     1.494498e-15
V7    -5.620335e-16
V8     1.149614e-16
V9    -2.414189e-15
V10    2.238554e-15
V11    1.724421e-15
V12   -1.245415e-15
V13    8.238900e-16
V14    1.213481e-15
V15    4.866699e-15
V16    1.436219e-15
V17   -3.768179e-16
V18    9.707851e-16
V19    1.036249e-15
V20    6.418678e-16
V21    1.628620e-16
V22   -3.576577e-16
V23    2.618565e-16
V24    4.473914e-15
V25    5.109395e-16
V26    1.686100e-15
V27   -3.661401e-16
V28   -1.227452e-16
dtype: float64

Whether a mean is "near zero" depends on how spread out the column is, so we also need each column's **standard deviation**: how far values typically sit from the mean.

In [27]:
v_stds = transactions[v_columns].std()
print(v_stds)

V1     1.958696
V2     1.651309
V3     1.516255
V4     1.415869
V5     1.380247
V6     1.332271
V7     1.237094
V8     1.194353
V9     1.098632
V10    1.088850
V11    1.020713
V12    0.999201
V13    0.995274
V14    0.958596
V15    0.915316
V16    0.876253
V17    0.849337
V18    0.838176
V19    0.814041
V20    0.770925
V21    0.734524
V22    0.725702
V23    0.624460
V24    0.605647
V25    0.521278
V26    0.482227
V27    0.403632
V28    0.330083
dtype: float64


Dividing each mean by its standard deviation shows how far off zero it is compared with the column's own spread. `abs()` drops the minus sign, so a mean below zero counts the same as one above, and `max()` gives the worst case across all 28.

Then a check of a claim in the data dictionary: PCA orders its columns from most variation (V1) to least (V28), so the standard deviations should shrink steadily. `is_monotonic_decreasing` is `True` if each value is no bigger than the one before it.

In [28]:
v_mean_to_std = v_means / v_stds
print(v_mean_to_std.abs().max())
print(v_stds.is_monotonic_decreasing)

7.386998835291376e-15
True


### Does any column never change, or barely change? (all 32 columns)

`nunique()` counts the different values in each column. A column with only 1 is constant and useless to a model. (`row_id` will have every value different, by design; it is never given to a model.)

In [29]:
print(transactions.nunique())

row_id    284807
Time      124592
V1        275663
V2        275663
V3        275663
V4        275663
V5        275663
V6        275663
V7        275663
V8        275663
V9        275663
V10       275663
V11       275663
V12       275663
V13       275663
V14       275663
V15       275663
V16       275663
V17       275663
V18       275663
V19       275663
V20       275663
V21       275663
V22       275663
V23       275663
V24       275663
V25       275663
V26       275663
V27       275663
V28       275663
Amount     32767
Class          2
dtype: int64


A column can have several values but still be almost all one value. For each column, this loop finds the **share of rows holding its single most common value**. `value_counts(normalize=True)` gives shares (0.25 = 25%) instead of counts (its default, `normalize=False`, gives counts), and `max()` picks the biggest share.

Any column above 0.95 (one value covering more than 95% of rows) is flagged as near-constant. That cutoff is a common rule of thumb, not a law. The `if` / `else` lines do the flagging: `if` runs its line only when the test is true, otherwise the `else` line runs. `Class` is expected to be flagged, because fraud is rare; it is the answer the model predicts, not a feature it learns from, so that is not a problem.

In [30]:
for column_name in transactions.columns:
    shares = transactions[column_name].value_counts(normalize=True)
    top_share = shares.max()
    if top_share > 0.95:
        print(column_name, top_share, "<-- near-constant")
    else:
        print(column_name, top_share)

row_id 3.5111496557317763e-06
Time 0.00012640138760634394
V1 0.0002703585234913468


V2 0.0002703585234913468
V3 0.0002703585234913468
V4 0.0002703585234913468
V5 0.0002703585234913468


V6 0.0002703585234913468
V7 0.0002703585234913468
V8 0.0002703585234913468
V9 0.0002703585234913468


V10 0.0002703585234913468
V11 0.0002703585234913468
V12 0.0002703585234913468
V13 0.0002703585234913468


V14 0.0002703585234913468
V15 0.0002703585234913468
V16 0.0002703585234913468


V17 0.0002703585234913468
V18 0.0002703585234913468
V19 0.0002703585234913468
V20 0.0002703585234913468
V21 0.0002703585234913468


V22 0.0002703585234913468
V23 0.0002703585234913468
V24 0.0002703585234913468
V25 0.0002703585234913468


V26 0.0002703585234913468
V27 0.0002703585234913468
V28 0.0002703585234913468
Amount 0.048060616487656554
Class 0.9982725143693799 <-- near-constant


## Step 7: Duplicate rows

Every row has its own unique `row_id`, so comparing *all* columns would find no duplicates at all. Every check here compares only the **original 31 columns**, leaving `row_id` out; `row_id` then shows which original transactions are the copies.

`+` between two lists joins them end to end. `feature_columns` is the 30 columns describing a transaction; `original_columns` adds `Class`.

In [31]:
feature_columns = ["Time"] + v_columns + ["Amount"]
original_columns = feature_columns + ["Class"]

print(len(feature_columns))
print(len(original_columns))

30
31


### Exact duplicates (all 31 original columns match)

Two counts, both useful. If one transaction appears 3 times:
- **Rows involved** = 3, every copy including the original (`keep=False`).
- **Extra copies** = 2, only the repeats after the first (`keep="first"`, the default). These are exactly the rows that would disappear if duplicates were removed keeping one of each.

`subset=` says which columns to compare. Its default is all columns, which would include `row_id` and find nothing.

In [32]:
is_exact_duplicate = transactions.duplicated(subset=original_columns, keep=False)
is_extra_copy = transactions.duplicated(subset=original_columns, keep="first")

print(is_exact_duplicate.sum())
print(is_extra_copy.sum())

1854
1081


Fraud versus genuine, for both counts (boolean filtering, then `value_counts()` on `Class`, both first used in Step 4). The fraud rows among the **extra copies** are exactly the fraud cases that removing duplicates would lose.

In [33]:
print("Rows involved in duplicates:")
print(transactions[is_exact_duplicate]["Class"].value_counts())
print()
print("Extra copies (would be removed):")
print(transactions[is_extra_copy]["Class"].value_counts())

Rows involved in duplicates:
Class
0    1822
1      32
Name: count, dtype: int64

Extra copies (would be removed):
Class
0    1062
1      19
Name: count, dtype: int64


### Contradictions: same 30 features, different Class

`shares_features` marks every row with a twin that has identical features, whatever its Class. `is_exact_duplicate` (above) marks rows with a twin identical in features **and** Class. A row in the first group but **not** the second has a feature twin, but none with its own Class: a real contradiction.

`&` means "and", `~` means "not". So `shares_features & ~is_exact_duplicate` reads: "has a feature twin, and does not have a twin with the same Class".

In [34]:
shares_features = transactions.duplicated(subset=feature_columns, keep=False)
is_contradiction = shares_features & ~is_exact_duplicate

print(shares_features.sum())
print(is_contradiction.sum())
transactions[is_contradiction].head()

1854
0


,row_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class


### The Step 6 clue: rows sharing V1 to V28 only

In Step 6 every V column had 9,144 fewer distinct values than rows. This counts extra copies of the **V1 to V28 values alone**, ignoring Time, Amount and Class. If it is bigger than the exact-duplicate extra copies, some transactions share identical V values but differ in Time or Amount. Those are not duplicates and are not removed; this is only to understand the clue.

In [35]:
v_only_extra_copy = transactions.duplicated(subset=v_columns, keep="first")
print(v_only_extra_copy.sum())

9144


## Step 8: Remove exact duplicate extra copies

**Removed:** the 1,081 extra copies of rows that match on all 31 original columns. One copy of each of the 773 duplicate groups is kept, so no distinct transaction is lost. Repeats would give some transactions extra weight in training and count the same case twice in evaluation.

**Kept:** every distinct transaction, including all 13 distinct fraud transactions involved in duplicates (there are no contradictions, so each fraud group's kept copy is itself fraud); the 8,063 rows that share V1 to V28 with another row but differ in Time or Amount (not duplicates); and the zero-amount rows.

First, save a record of exactly which rows are removed, by `row_id`.

In [36]:
removed_rows = transactions[is_extra_copy]
removed_rows.to_csv("../data/staging/removed_duplicate_rows.csv", index=False)
print(len(removed_rows))

1081


Remove them: `~is_extra_copy` flips every `True` to `False` and back, so filtering with it keeps every row that is **not** an extra copy. Using the same `True`/`False` column we counted in Step 7 guarantees the rows removed are exactly the ones we inspected.

Then check it worked: rows removed should be 1,081, a fresh duplicate check should find 0 extra copies, fraud rows removed should be 19, and `row_id` should still be unique.

In [37]:
transactions_clean = transactions[~is_extra_copy]

rows_removed = len(transactions) - len(transactions_clean)
print(rows_removed)

remaining_extra_copies = transactions_clean.duplicated(subset=original_columns, keep="first")
print(remaining_extra_copies.sum())

# Class is 1 for fraud and 0 for genuine, so adding up the column counts the fraud rows.
fraud_removed = transactions["Class"].sum() - transactions_clean["Class"].sum()
print(fraud_removed)

print(transactions_clean["row_id"].is_unique)

1081


0
19
True


Save the cleaned data to `data/processed`. This is the last change Phase 1 makes to the data, so this is the one final file every later phase uses. `index=False` as before (explained in Step 2).

In [38]:
transactions_clean.to_csv("../data/processed/creditcard_clean.csv", index=False)
print(transactions_clean.shape)

(283726, 32)


## Step 9: The real class balance on the cleaned data

Load the saved clean file itself, so the numbers describe exactly the file every later phase uses.

Then check the saved file matches what we saved. `equals()` is `True` only if both tables have the same columns, types and every single value. `reset_index(drop=True)` renumbers `transactions_clean`'s rows from 0, because it still carries its old position numbers with gaps where duplicates were removed, while the reloaded file is numbered fresh. `drop=True` throws the old numbers away instead of keeping them as an extra column (the default, `drop=False`). Our real ID is `row_id`, so nothing is lost.

In [39]:
transactions_final = pd.read_csv("../data/processed/creditcard_clean.csv")

print(transactions_clean.reset_index(drop=True).equals(transactions_final))

True


Counts, percentages, and "one fraud for every how many genuine". `class_counts[0]` picks out the genuine count and `class_counts[1]` the fraud count. Multiplying the shares by 100 turns them into percentages.

In [40]:
class_counts = transactions_final["Class"].value_counts()
print(class_counts)
print()

class_shares = transactions_final["Class"].value_counts(normalize=True)
print(class_shares * 100)
print()

genuine_per_fraud = class_counts[0] / class_counts[1]
print(genuine_per_fraud)

Class
0    283253
1       473
Name: count, dtype: int64

Class
0    99.83329
1     0.16671
Name: proportion, dtype: float64

598.8435517970402


The same counts on the data from before cleaning (`transactions`, from Step 1), for comparison.

In [41]:
print(transactions["Class"].value_counts())
print()
print(transactions["Class"].value_counts(normalize=True) * 100)

Class
0    284315
1       492
Name: count, dtype: int64

Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64
